## ESPnet TTS Demonstracija

### 0. Instaliuojame bibliotekas

In [ ]:
# !apt-get update
!apt-get install -y espeak-ng

!pip install "espnet[tts] @ git+https://github.com/airenas/espnet.git@demo.v01" phonemizer -q --disable-pip-version-chec
!pip install parallel_wavegan -q --disable-pip-version-check

print("\nDONE: paruošta")

### 0.1 Pataisome PWGan dėl Python 3.12

In [ ]:
## workaround: fix parallel_wavegan for python3.12
import scipy.signal.windows
import sys
sys.modules['scipy.signal'].kaiser = scipy.signal.windows.kaiser

### 1. Inicializuojame ESPnet ir pagalbines funkcijas

In [ ]:
from espnet2.bin.tts_inference import Text2Speech
import soundfile as sf
from IPython.display import Audio, HTML
from huggingface_hub import snapshot_download
from pathlib import Path

device = "cpu"
device = "cuda" # uncomment if you have a GPU and want to use it

def download_vocoder(tag: str) -> str:
    repo_dir = snapshot_download(tag)
    repo_path = Path(repo_dir)

    for file in repo_path.rglob("*.pkl"):
        # print(f"found {str(file)}")
        return str(file)

    raise FileNotFoundError("No *.pkl file found in the vocoder repo")
   
class TTSData:
    def __init__(self, tts, info):
        self.tts = tts
        self.info = info


def init_tts(am_tag=None, vocoder_tag=None, info=""):
    voc_str = vocoder_tag
    if not voc_str:
        voc_str = "Griffin-Lim"
    
    print(f"\n===================================\nAM      = {am_tag}")
    print(f"Vocoder = {voc_str}")
    print(f"===================================")

    vocoder_file=None
    if vocoder_tag:
        vocoder_file = download_vocoder(vocoder_tag)
        vocoder_tag = None
    
    tts = Text2Speech.from_pretrained(
        model_tag=am_tag,
        vocoder_tag=vocoder_tag,
        vocoder_file=vocoder_file,
        device=device
    )
    return TTSData(tts, info)

print ("\nDONE: ESPnet paruoštas")    

### 2. Parsiunčiame/užkrauname modelius iš HuggingFace

Paruošti tokie modeliai:
1. AM vyriškas balsas: [VSSA-SDSA/sing-arn.fastspeech2.v01](https://huggingface.co/VSSA-SDSA/sing-arn.fastspeech2.v01)
2. AM moteriškas balsas - [VSSA-SDSA/sing-agn.fastspeech2.v01](https://huggingface.co/VSSA-SDSA/sing-agn.fastspeech2.v01)
4. Vokoderis vyriškas balsas - [VSSA-SDSA/sing-arn.vocoder.style_melgan.v01](https://huggingface.co/VSSA-SDSA/sing-arn.vocoder.style_melgan.v01)
5. Vokoderis moteriškas balsas - [VSSA-SDSA/sing-agn.vocoder.style_melgan.v01](https://huggingface.co/VSSA-SDSA/sing-agn.vocoder.style_melgan.v01)


In [ ]:
am_tag_hf_arn="VSSA-SDSA/sing-arn.fastspeech2.v01"
vocoder_tag_hf_arn= "VSSA-SDSA/sing-arn.vocoder.style_melgan.v01"

tts_data = init_tts(am_tag=am_tag_hf_arn, vocoder_tag=vocoder_tag_hf_arn, info="arn-fastspeech2+arn-style")
tts = tts_data.tts

print (f"\nDONE: modelis paruoštas\n")  

### 3. Pasirenkame tekstą


In [ ]:
from ipywidgets import FileUpload
from IPython.display import display, Audio
import io
import soundfile as sf

uploader = FileUpload(accept='.txt', multiple=False)
display(uploader)


In [ ]:
#################################################################
# Tekstas iš pasirinkto failo
#################################################################
val = uploader.value
if isinstance(val, dict):
    uploaded_files = list(val.values())
elif isinstance(val, (list, tuple)):
    uploaded_files = val
else:
    raise RuntimeError(f"Unexpected uploader.value type: {type(val)}")

if len(uploaded_files) == 0:
    raise RuntimeError(f"Select txt file")
uploaded_file = uploaded_files[0]  
txt = bytes(uploaded_file['content']).decode("utf-8")
l = len(txt)
print(f"text: {txt[0:50]} ...({l})... {txt[l-50:l]}")


In [ ]:
#################################################################
# Suskaidome dalimis
#################################################################
import re
parts = []
current = ""
wanted_chars = 150
for s in re.findall(r'[^.!?]*[.!?]', txt):
    s = s.strip()
    if len(current) + len(s) <= wanted_chars:
        current += s
    else:
        parts.append(current.strip())
        current = s

if current:
    parts.append(current.strip())
    
print(f"parts: {len(parts)}")
for i, s in enumerate(parts[0:3]):
    print(f"{i}: {s}")    

### 4. Rezultatas

Sintezuojamas tekstas, saugome wav po maždaug 2 min

In [ ]:
#################################################################
# Saugome audio į part_0.wav, part_1.wav, part_2.wav, .... 
#################################################################
import numpy as np
from tqdm import tqdm
import shutil
import os

max_sec = 120
buf, dur, idx = [], 0, 0

audio_dir = Path("audio_parts")
if audio_dir.exists():
    shutil.rmtree(audio_dir)
os.makedirs(audio_dir, exist_ok=True)


def flush():
    global buf, dur, idx
    if not buf: return
    fn = audio_dir / f"part_{idx}.wav"
    sf.write(fn, np.concatenate(buf), tts.fs)
    print(f"\n==========================================================================\n")
    display(Audio(fn))
    print(f"WAV {idx}")
    buf, dur, idx = [], 0, idx + 1

for text in tqdm(parts):
    wav = tts(text)["wav"].cpu().numpy()
    d = len(wav) / tts.fs

    if dur + d > max_sec:
        flush()

    buf.append(wav)
    dur += d

flush()
print(f"\nDONE\n")

In [ ]:
### Download all created files as zip
import zipfile
import os
from IPython.display import FileLink

zip_name = audio_dir / "all_audio_parts.zip"

with zipfile.ZipFile(zip_name, 'w') as z:
    for file in os.listdir(audio_dir):
        if file.startswith("part_") and file.endswith(".wav"):
            print(f"adding {audio_dir / file}")
            z.write(audio_dir / file)

print("ZIP created:", zip_name)

display(FileLink(zip_name))